# ProteinMPNN supercharging benchmark, updated figures

PLAN.md Section 11.4. This notebook is a second view of the same results, not a
replacement for `analysis.ipynb`. It carries ten figures over the nine-arm set,
and it writes to `figures/updated/` so nothing in `figures/` is overwritten.

**Carried here:** F1, F2, F3, F4, F5, F6, F7, F8, F9 and the new F10.
**Not carried here:** F1, F2, F2_inset, F8, F8alt, F11, F13 and every table.
Those stay in `analysis.ipynb` and are not regenerated.

**Three arms are new** relative to `analysis.ipynb`: `mpnn_hyper` and
`mpnn_halo`, the same supercharging decoder over the HyperMPNN and HaloMPNN
checkpoints, and `rosetta_hbond_off`, the Rosetta baseline with h-bonded
sidechain protection turned off so it is matched to the primary MPNN arm.
A tenth series, the unguided ProteinMPNN rejection pool, appears on F1, F2 and
F8.

**Titles carry no context here.** In `analysis.ipynb` several figure titles hold
whole caveat sentences. Every title below is short and the context sits in a
**Notes** cell under each figure. Nothing was dropped; the caveats that lived
only in a title are written out at greater length.

**This notebook reads only `results/*.csv` and `data/scaffold_manifest.csv`, and
writes only `figures/updated/`.** It does not touch the cluster, import torch or
pyrosetta, or recompute a metric. If a number appears here, a script in
`scripts/` computed it from a file on disk and wrote it to a CSV.

### Figure numbering

Figures here are numbered **F1 to F10 in the order they are plotted below**, and
written to `figures/updated/`. That is a different scheme from `analysis.ipynb`,
which keeps its original F1 to F14 in `figures/`. The two sets live in separate
directories and never overwrite each other, but a reference to "F4" is ambiguous
without saying which notebook, so this table is the key.

| Here | Content | Was, in `analysis.ipynb` |
|---|---|---|
| F1 | Charge ladder against the unguided sampling distribution | F3 |
| F2 | Mutational efficiency | F4 |
| F3 | Sequence diversity | F5 |
| F4 | Energetic cost | F6 |
| F5 | Per-term energy decomposition | F7 |
| F6 | Hydrogen bonds gained and lost | F9 |
| F7 | Independent structure prediction | F10 |
| F8 | Runtime per accepted design | F12 |
| F9 | Where predicted confidence starts to fall | F14 |
| F10 | Charge demand, decoding temperature, predicted confidence | new |

`analysis.ipynb`'s F1, F2, F2_inset, F8, F8alt, F11 and F13 have no counterpart
here and are not renumbered; they are unchanged in `figures/`.


## 1. Setup, data load, schema assertions

In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd
# Imported explicitly rather than relied on as a notebook builtin, so the
# same cells run under scripts/run_notebook.py, which execs them in a plain
# namespace with no IPython builtins injected.
from IPython.display import display

import analysis_updated_lib as U

U.set_style()
data = U.load_updated()

In [ ]:
# Coverage. Nothing below silently plots an empty panel: if a column is absent
# for an arm, that arm is missing from the figure that needs it, and this table
# is where you find out why.
d = data["designs"]
coverage = pd.DataFrame({
    "designs": d.groupby("method").size(),
    "threaded (d_reu_per_res)": d.groupby("method")["d_reu_per_res"].count(),
    "predicted (plddt_mean)": d.groupby("method")["plddt_mean"].count(),
    "final_temperature": d.groupby("method")["final_temperature"].count(),
    "scaffolds": d.groupby("method")["scaffold_id"].nunique(),
}).reindex(U.METHOD_ORDER).dropna(how="all")
coverage

In [ ]:
# Assertions, not comments. Each one is a claim the figures rely on.
missing = [m for m in U.METHOD_ORDER if m not in set(d["method"].unique())]
assert not missing, f"arms missing from designs.csv: {missing}"

# The h-bond convention reads backwards from the flag name. PLAN.md Section 5:
# hbond_filter True means protection was ON.
for arm, expected in (("mpnn_soluble", False),
                      ("mpnn_soluble_hbond_protected", True),
                      ("mpnn_hyper", False),
                      ("mpnn_halo", False),
                      ("rosetta", True),
                      ("rosetta_hbond_off", False)):
    got = d.loc[d["method"] == arm, "hbond_filter"].dropna().unique()
    assert list(got) == [expected], f"{arm}: hbond_filter {got}, expected {expected}"

# The biased arms must be distinguishable from mpnn_vanilla_weights in the
# results. They run with --weights original on the command line because the
# shim reuses that directory name, so the weights column is the only record of
# which checkpoint produced the row.
for arm, label in (("mpnn_hyper", "hyper"), ("mpnn_halo", "halo")):
    got = d.loc[d["method"] == arm, "weights"].dropna().unique()
    assert list(got) == [label], f"{arm}: weights {got}, expected [{label!r}]"

assert (d["status"] == "ok").all() or d["status"].value_counts().to_dict()
print("status:", d["status"].value_counts().to_dict())
print(f"{len(d)} designs, {d.method.nunique()} arms, {d.scaffold_id.nunique()} scaffolds")

## 2. Where the charge ladder sits (F1)

In [ ]:
fig = U.fig1(data)
U.save(fig, "F1_charge_ladder_vs_unguided")

### Notes on F1

**What it shows.** One row per scaffold, ordered by the mean net charge of its
2,000 unguided ProteinMPNN samples. The violet band is that sample distribution,
mean ± 1.96 sd. The diamond is the wild-type net charge. The eight marks per row
are the ladder targets: an open green circle where at least one of the 2,000
samples landed exactly on it, a red cross where none did.

**Why the form changed.** The previous version drew capped error bars, two
marker types and a legend inside the axes on 25 rows, which put the legend on
top of the data at the charge extremes. The spread is now a band, the legend is
outside, and the horizontal gridlines are off, since the y axis is categorical
and the lines added nothing.

**Two fixes carried in.** The legend previously said "0 of 2000 sampled" as a
literal; it now reads the depth from `rejection_curve.csv`'s `n_samples_drawn`,
so it cannot go stale if the pools are ever rerun at a different depth. And the
wild-type entry in the legend used the pool's colour while the markers on the
plot were drawn white-faced; the swatch now matches what is drawn.

**What it is evidence for.** The red crosses are the answer to R3. Where a
target sits outside the sampled mass, unguided sampling does not reach it
rarely, it does not reach it at all in 2,000 draws.

## 3. Mutational efficiency (F2)

In [ ]:
fig = U.fig2(data)
U.save(fig, "F2_mutational_efficiency")

In [ ]:
# The unguided sampler, reported rather than drawn. Two limits are visible in
# the first table: zero on-target samples at +16 and +24 across all 25
# scaffolds while every negative rung has some, and a cell contributes only
# where the pool hit at least once, 81 of 200.
display(U.unguided_summary(data))
display(U.unguided_by_fold_class(data))

### Notes on F2

**What it shows.** Mutations spent per unit of charge actually moved, lower is
better, for the nine guided arms. Facets are the three CATH fold classes in a
fixed order, then all 25 scaffolds pooled.

**The unguided pool is not drawn here, and that is deliberate.** It spends 8 to
12 mutations per unit charge against roughly 0.8 for every guided arm. On a
shared linear axis it flattens all nine guided boxes and both Der reference
lines into one indistinguishable strip, and on a log axis fitted to it the
guided arms lose the resolution this figure exists to provide. F1 already makes
the unguided sampler's behaviour its entire subject. The measurement is kept,
not the series: `results/rejection_hits.csv` still holds all 6,428 on-target
samples and the table below reports them per fold class.

**Facet order.** Fixed to `mainly_alpha`, `mainly_beta`, `alpha_beta`, then all
25 pooled, replacing an alphabetical `sorted()` that also gave eGFP a facet of
its own. eGFP enters outside the CATH split and has no fold class; a facet
holding one scaffold reads badly beside facets holding eight, so it is carried
by the pooled facet only.

**Reference lines.** Der et al. 2013 report 0.6 mutations per unit charge for
AvNAPSA and 0.85 for Rosetta. Both are drawn with real legend handles.


## 4. Sequence diversity (F3)

In [ ]:
fig = U.fig3(data)
U.save(fig, "F3_sequence_diversity")

### Notes on F3

**What it shows.** The four Section 10.2 diversity metrics, one panel each, per
scaffold × target cell, all restricted to designable positions.

**Read the AvNAPSA column with its history.** That arm originally ran at
`nstruct 1`, which made its unique-set count 1 by construction and its pairwise
Hamming and positional entropy undefined rather than zero. It was rerun at
`nstruct 10` on the author's instruction (PLAN.md Section 0.1 item 16), so those
numbers are now observations. The measured result is that AvNAPSA is close to
deterministic in practice: 150 of 200 cells return a single mutation set.

**Rosetta is not near-deterministic**, contrary to what PLAN.md Section 6
predicted before the arm ran. It returns a median of 7 distinct mutation sets
out of 10. The diversity comparison has to be stated on that measurement, not on
the prediction.

**`mpnn_soluble` returns all 10 distinct in 192 of 200 cells.** The two biased
arms and the h-bond-protected arm are drawn beside it here for the first time;
whether the biased checkpoints trade diversity for anything is what the first
two panels are for.

## 5. Energetics (F4, F5)

In [ ]:
fig = U.fig4(data)
if fig is not None:
    U.save(fig, "F4_energetic_cost")

### Notes on F4

**What it shows.** Median Rosetta energy change per residue against charge
demand, interquartile ribbon across designs, one line per arm.

#### Why the curves look like this

**The rise at the extremes is real within a scaffold, but the extreme bins are
one scaffold.** Threading is tiered: Tier B covers all 25 scaffolds at ±8 and
±16 only, and Tier A covers eGFP at every rung. So ±4 and ±24 are eGFP alone,
90 designs each, while ±8 and ±16 pool 25 scaffolds. Comparing across the whole
axis therefore mixes a change in charge demand with a change in scaffold panel.
Holding the scaffold fixed removes the confound and the effect survives: within
eGFP, median ΔREU per residue for `mpnn_soluble` runs 0.013, 0.010, 0.023, 0.122
across |ΔQ density| 4, 8, 16, 24. Cost rises with demand, and it rises steeply
only at the top rung.

**The ordering between arms is the expected quality gradient.** At eGFP |24| the
random control is worst at 0.273, AvNAPSA and the two biased-weight arms sit
between 0.186 and 0.209, and `mpnn_soluble` is lowest at 0.122. The biased
checkpoints costing more than the soluble one at high demand is the same result
F10 shows from the other direction: they resist charge-directed decoding, need
more temperature to reach the target, and end up in less favourable sequence
space when pushed.

**Both Rosetta arms sit below zero everywhere, and that is not evidence that
they build better proteins.** Three things produce it, none of them a property
of the designs:

1. **The mover optimises the quantity being plotted.**
   `compare_residue_energies_mut(True)` makes Supercharge reject a candidate
   mutation whose per-residue energy is worse, scored with the same `ref2015`
   function used here. The arm is being graded on its own objective.
2. **It stops early rather than paying.** Both Rosetta arms reach their exact
   target in only 24 to 34% of threaded cells, against 100% for every MPNN arm.
   A run that stops when it runs out of acceptable mutations moves less charge
   and makes fewer mutations, so it accumulates less deviation from the wild
   type.
3. **Its cost does not grow with mutation count.** Correlation between
   `d_reu_per_res` and `n_mutations` is 0.07 for `rosetta`, against 0.37 to 0.83
   for every other arm. A method that filters each mutation on this score
   accumulates no cost as mutations accumulate, which is exactly the flat
   negative line the figure shows.

A fair reading is that F4 compares arms on how much energetic ground they give
up **per unit of charge they actually deliver**, and Rosetta's advantage here is
inseparable from its failure to deliver the charge. F2 and the hit rates are
where that trade is visible.

**The reference relaxes unrestricted, which biases every delta upward.** The
wild-type reference is one relaxed structure per scaffold per tier, built by
this benchmark because the repo's `threading_only.py` does not relax a
zero-mutation sequence; referencing against the unrelaxed crystal would have
folded the whole relaxation gain, a median 305 REU across the panel, into every
delta. But the reference relaxes every residue while each design relaxes only
within a shell around its mutations, so the comparison is tilted against the
designs and values near zero should not be read as "free". `logs/ISSUES.md`
item 11 carries both the relaxed and crystal scores.

**Coverage.** 2,880 of 15,120 designs are threaded. The Section 1 coverage table
gives the count per arm.


In [ ]:
fig = U.fig5(data)
if fig is not None:
    U.save(fig, "F5_energy_decomposition")

### Notes on F5

**What it shows.** The median change in each Rosetta score term, grouped by
term, one bar per arm. Deliberately styled as an analogue of Der et al. 2013
Figure 5 so a referee who knows that figure can read this one immediately.

**Six terms, not eight.** PLAN.md Section 8 names `fa_dun` and `p_aa_pp` as
well, but the frozen `designs.csv` schema in Section 10.1 carries only
`d_fa_atr`, `d_fa_rep`, `d_fa_sol`, `d_fa_elec`, `d_hbond_sc` and
`d_hbond_bb_sc`. Adding the other two is a schema change and goes through
PLAN.md first, so the figure draws what exists rather than an approximation of
what was specified.

**`ref2015` is the score function throughout.** Values are medians, not means:
the threaded set contains a small number of designs whose repulsive term is very
large, and a mean would report those rather than the arm.

## 6. Hydrogen bonding (F6)

In [ ]:
fig = U.fig6(data)
if fig is not None:
    U.save(fig, "F6_hydrogen_bonds")

### Notes on F6

**What it shows.** Side-chain hydrogen bonds gained and lost against the matched
relaxed wild type, counted with Der et al.'s thresholds: strong at ≤ −0.5 REU,
weak between −0.5 and −0.1. Backbone-to-backbone bonds are excluded; they are a
property of the fixed scaffold, not of the surface mutations.

**The third panel is new, and it exists because the old comparison was
confounded.** `scripts/lib/baseline.py` hardcoded
`dont_mutate_hbonded_sidechains(True)`, so the Rosetta arm protected h-bonded
sidechains while the primary MPNN arm, which passes `-mhbond`, did not. The two
were compared anyway. The added `rosetta_hbond_off` arm sets that switch False,
which is exactly what `-mhbond` does on the MPNN side, so the panel now shows
one design method at a time with protection on and off.

**All four series in that panel sit on the same scaffolds.** The two control
arms are configured to the same subset, resolved once by `lib/io.control_subset`
so the two cannot drift, and the panel additionally intersects the scaffolds
actually present on all four arms rather than assuming the configuration held.
The panel title reports how many scaffolds survived that intersection.

**Panel c reads by linestyle, not by colour.** Solid is protection off, dotted
is protection on, and the two arms keep the colours they carry in the other
panels. On the measured data the protected arms sit above their unprotected
counterparts at high charge demand on both design methods, which is the
direction Der et al. would predict.

**What the manuscript assumed.** The text assumes h-bond filtering can be left
off because ProteinMPNN "can implicitly account for such interactions." That
assumption sits directly opposite Der et al.'s central finding about AvNAPSA
destroying surface hydrogen bonds, and this figure is where it is tested rather
than restated.

## 7. Independent structure prediction (F7)

In [ ]:
fig = U.fig7(data)
if fig is not None:
    U.save(fig, "F7_structure_prediction")

### Notes on F7

**What it shows.** ESMFold2 on every design, faceted by arm. Three rows:
absolute pLDDT, the change in pLDDT against each scaffold's own predicted wild
type, and CA RMSD to the wild-type **crystal** chain after TM-align
superposition. RMSD is never measured against the threaded model.

**The absolute row is new.** The previous version showed only the delta. The two
answer different questions: the absolute row asks whether a design is still
confidently folded at all, the delta asks what the design cost relative to a
starting point that varies a lot across the panel. The dotted line on the top
row is pLDDT 80, and F9 panel b is the reason it is drawn as context rather
than as a threshold.

**AF3 is removed.** It carried 32 designs across 4 of the 9 arms, so the overlay
was absent from more than half the facets and could not support a comparison
across them. The AF3 subset is unchanged on disk and `T4_af3_subset` in
`analysis.ipynb` remains its artifact. Its own caveat still stands: 32 of the 36
AF3 predictions are below pLDDT 80 in single-sequence mode.

**Faceted rather than overlaid** because a scatter-shaped form puts arbitrary
pairs of series side by side, and the palette is certified for neighbouring
pairs only. See the module docstring for the measured numbers.

**ESMFold used the `-Fast` model** for the full sweep.

## 8. Runtime (F8)

In [ ]:
fig = U.fig8(data)
U.save(fig, "F8_runtime")

In [ ]:
# How much of the unguided box is censored away, computed rather than typed.
vals, censored, total = U.unguided_cost_per_hit(data["curve"])
print(f"unguided cells plotted: {len(vals)} of {total}; "
      f"{censored} censored as a lower bound because the pool never hit")
print(f"median unguided CPU-seconds per on-target design: {pd.Series(vals).median():.2f}")

### Notes on F8

**What it shows.** CPU-seconds per on-target design, log scale, retries
included. One box per arm plus the unguided rejection sampler.

**Everything is CPU.** PLAN.md Section 0.2 D resolved the partition conflict by
timing every method on `main` with the same 4-CPU allocation, so the arms are
comparable. `rejection_curve.csv` still names its columns
`gpu_seconds_per_sample` and `expected_gpu_seconds_per_hit`; those hold CPU
seconds under legacy names, and renaming them is a schema change that would go
through PLAN.md first.

**Both sides of this figure are censored, and both censorings flatter the same
direction.** A guided cell with no on-target design has no cost per hit and is
excluded. An unguided cell whose pool never hit has only a lower bound, printed
in the CSV as `>N`, and is excluded too. The cell above prints how many. The
unguided box therefore describes only the cells where unguided sampling worked
at all, which makes it a lower bound on the true cost, and the gap between it
and the guided arms is a lower bound on the gap.

**The random control stretches the axis over seven decades.** It is pure
Python with no model in the loop, so it lands near 4e-5 s and leaves five empty
decades between it and everything else. That gap is the measurement, not a
plotting artefact, and dropping the series to tighten the axis would remove a
real comparison to make the figure look better.

**Runtime is not the argument.** F2 and F1 in `analysis.ipynb` make the
argument: at the extreme rungs the unguided sampler does not reach the target at
any budget, so a seconds-per-hit number does not exist there to plot.

## 9. Predicted confidence (F9, F10)

In [ ]:
fig = U.fig9(data)
if fig is not None:
    U.save(fig, "F9_plddt_dropoff")

### Notes on F9

**The question behind this figure** was posed as "at what temperature or
mutation count does pLDDT drop below 0.8". Two things stop it being answerable
in that form, and both are shown rather than worked around.

**First, the absolute cutoff mostly reports which scaffold was picked.** Panel b
is that fact directly: a large fraction of the wild types already predict below
80 before any design is made. So every summary in panel a is stated against each
scaffold's own predicted wild type. `d_plddt_vs_wt_pred` is that difference and
is computed in Phase 6, not here; the wild-type value itself is not stored as a
column and `analysis_lib.plddt_frame` reconstructs it by subtraction.

**Second, the mutation-count axis is unbalanced.** Only the longest scaffolds
reach the high bins, and those are also the ones ESMFold predicts most
confidently, so pooling designs makes the curve turn upward past 30 mutations
and report the scaffold panel rather than the mutations. Panel a is therefore
the median of per-scaffold medians with the interquartile range taken across
scaffolds, and a bin holding fewer than `analysis_lib.MIN_SCAFFOLDS_PER_BIN`
scaffolds is dropped rather than drawn thin.

**Reduced from four panels to two.** The |ΔQ density| panel said the same thing
as panel a on a coarser axis. The decoding-temperature panel is promoted to F10,
where it sits next to the charge demand that drove the temperature up.

**One cosmetic fix.** Panel b's below-threshold highlight used to reuse the
AvNAPSA method colour, which read as a method reference immediately above a
method legend. It is a neutral red now; there are no methods in that panel.

In [ ]:
fig = U.fig10(data)
if fig is not None:
    U.save(fig, "F10_temperature_and_confidence")

In [ ]:
# The two claims the F10 notes make, read off the data rather than typed.
U.temperature_confound(data)

### Notes on F10

**What it shows, left to right.** Panel a: the decoding temperature a target
cost, against how much charge was demanded. Panel b: the change in predicted
confidence against that temperature. Panel c: the absolute predicted confidence
against the same axis. All three use the same balancing as F9, median of
per-scaffold medians with the spread across scaffolds and thin bins dropped.

**MPNN arms only.** `final_temperature` is written by the `-u` escalation loop
in `protein_mpnn_supercharge.py`, which raises the sampling temperature in 0.1
steps from 0.3 to a 0.9 ceiling until the target is hit. The classical arms and
the random control have no such knob and the column is blank for them, so
plotting them here would be inventing an x coordinate.

**Panels b and c cannot isolate temperature.** Temperature and mutation count
rise together: in the table above, the median mutation count climbs steadily
across the temperature bins. A design at T = 0.9 is both hotter and more heavily
mutated than one at T = 0.3, and nothing in this figure separates the two. F9
panel a is the mutation-count view of the same designs and the two should be
read together. Neither supports a causal claim about temperature.

**The escalation is rare, so the high-temperature bins are thin.** Most cells
never leave 0.3. The `n_scaffolds` column in the table above is what the
5-scaffold rule is applied to, and it is why the per-arm lines stop before the
top of the axis. The dashed line pools every MPNN design so the high end is
representable; it is not an arm and is labelled as pooled.

**The raw pooled medians are non-monotone** in absolute pLDDT across the
temperature bins. That is the scaffold panel showing through, the same effect
F9 documents on the mutation axis, and it is exactly why the balancing is
applied rather than the pooled numbers being plotted.

**Escalation is a capacity signal, not only a confidence one.** Panel a is the
cleanest per-arm comparison in this notebook: an arm that reaches the same
target at a lower temperature needed less help from the sampler to get there.

## 13. Draft text

Paragraphs for reuse, reporting what was run and what the numbers are. **The
conclusions are not drawn here.** Every claim below is a measurement with its
spread or its test attached, and the reader decides what it means. Numbers are
quoted from the CSVs this notebook reads; where a figure is cited it is the
updated numbering in the table at the top.

### Methods

**Design arms.** Nine arms were run over a common grid of 25 scaffolds, 8
charge-ladder rungs and 10 samples per cell. Five drive the modified ProteinMPNN
decoder, which selects charged substitutions position by position until a target
net charge is reached: `mpnn_soluble` (SolubleMPNN weights, the primary arm),
`mpnn_vanilla_weights` (stock ProteinMPNN weights), `mpnn_soluble_hbond_protected`
(primary arm with h-bonded sidechains protected from mutation), and `mpnn_hyper`
and `mpnn_halo`, which are the primary arm's decoder over the HyperMPNN and
HaloMPNN checkpoints respectively. Three are classical baselines driven through
PyRosetta's `Supercharge` mover: `avnapsa` (sequence-based surface definition),
`rosetta` (score-based, h-bonded sidechains protected) and `rosetta_hbond_off`
(score-based, protection released so that it matches the primary MPNN arm's
treatment). `random_control` mutates randomly chosen designable positions to
charged residues until the target is met. Two arms, `mpnn_soluble_hbond_protected`
and `rosetta_hbond_off`, run on a 7-scaffold control subset rather than the full
panel; the subset is resolved once from the scaffold manifest so both land on
identical scaffolds. All other arms are complete at 2,000 designs, giving 15,120
designs in total, every one with `status="ok"`.

**Alternative checkpoints.** HyperMPNN and HaloMPNN share the `v_48_020`
architecture (118 state-dict tensors, 48 edges, noise level 0.2), so they are
loaded by the unmodified supercharging script through a directory laid out the
way that script expects, holding symlinks to the checkpoint and to the patched
`protein_mpnn_utils`. The script itself is not modified. Each design row records
which checkpoint produced it in the `weights` column.

**Decoding.** All MPNN arms sample at temperature 0.3 with the unrestrict flag
set, which raises the temperature in 0.1 steps to a ceiling of 0.9 when a target
is missed, and record the temperature at which the target was finally reached.
Seeds are derived per cell from a fixed base seed and a digest of scaffold, arm
and target charge, and are recorded in every output row.

**Threading and energetics.** A tiered subset of designs was threaded onto its
backbone and relaxed with PyRosetta: Tier A covers the focus scaffold eGFP at
every rung with 10 samples and 5 relax cycles, Tier B covers all 25 scaffolds at
±8 and ±16 with 3 samples and 2 relax cycles. 2,880 designs of 15,120 carry
energetics. Energies are `ref2015` deltas against a relaxed wild-type reference
built per scaffold per tier; hydrogen bonds are counted with Der et al.'s
thresholds, strong at ≤ −0.5 REU and weak between −0.5 and −0.1, excluding
backbone-to-backbone bonds.

**Structure prediction.** ESMFold2 was run on all 15,120 designs plus 25
wild-type sequences. RMSD and TM-score are computed after TM-align
superposition onto the wild-type crystal chain, never onto the threaded model.
ΔpLDDT is stated against each scaffold's own predicted wild type.

**Unguided reference.** Unmodified ProteinMPNN was sampled 2,000 times per
scaffold at the same temperature and over the same designable set, with no
charge direction, and the resulting sequences filtered post hoc for those
landing exactly on a ladder target. This gives 50,000 samples, of which 6,428
are on target across 81 of the 200 scaffold-target cells.

**Statistics.** Every test pairs `mpnn_soluble` against one other arm on cells
sharing a scaffold and a target, so the unit of analysis is the cell, not the
design. Effect sizes accompany every test: the median paired difference with a
BCa bootstrap 95% CI, and Cliff's delta. p-values are Holm-corrected within
metric. 728 tests were run, 223 significant at Holm-corrected p < 0.05.

### Results

**Charge targeting.** All five MPNN arms reach their exact target in 2,000 of
2,000 designs. The classical baselines do not: `avnapsa` reaches it in 59.3%,
`rosetta` in 23.2% and `rosetta_hbond_off` in 28.9%. Releasing h-bond protection
on the Rosetta arm raises its hit rate on the shared 7-scaffold subset by a
median 0.70 per cell against the primary MPNN arm's rate (95% CI 0.60 to 0.80,
Holm p < 0.0001, Cliff's delta 0.98). Unguided sampling reaches the target in 81
of 200 cells and in zero cells at the two most positive rungs, +16 and +24,
while every negative rung including −24 has on-target samples; F1 shows the
targets that fall outside the sampled mass (Section 4 of this notebook reports
the per-rung counts).

**Mutational efficiency.** Median mutations per unit of charge moved is 0.82 for
`mpnn_soluble`, 0.80 for `mpnn_hyper`, 0.82 for `mpnn_halo`, 0.81 for
`mpnn_vanilla_weights`, 0.60 for `avnapsa`, 0.75 for `rosetta` and 0.72 for
`rosetta_hbond_off` (F2). The Der et al. 2013 reference values, 0.6 for AvNAPSA
and 0.85 for Rosetta, are drawn on the figure. The unguided sampler spends 2.5
to 13.7 mutations per unit charge depending on the rung, an order of magnitude
more, and is reported as a table rather than drawn because it flattens the axis.

**Sequence diversity.** Out of 10 samples per cell, all 10 mutation sets are
distinct in 192 of 200 cells for `mpnn_soluble`, 182 for `mpnn_vanilla_weights`,
157 for `mpnn_hyper` and 147 for `mpnn_halo`. `avnapsa` returns a single set in
150 of 200 cells. `rosetta` returns a median of 7 distinct sets and reaches all
10 in only 25 of 200 cells, which contradicts the near-determinism the plan
predicted for it before the arm ran. Against
`mpnn_soluble`, mean pairwise Hamming distance is lower for `mpnn_halo` by a
median 2.68 (95% CI 2.02 to 3.21, Holm p < 0.0001) and for `rosetta_hbond_off`
by 6.21 (95% CI 4.76 to 7.08, Holm p < 0.0001).

**Energetic cost.** Median ΔREU per residue rises with charge demand within a
fixed scaffold: for `mpnn_soluble` on eGFP it runs 0.013, 0.010, 0.023 and 0.122
across |ΔQ density| 4, 8, 16 and 24. At the highest demand on that scaffold the
random control is highest at 0.273 and `mpnn_soluble` lowest at 0.122, with
`avnapsa`, `mpnn_hyper` and `mpnn_halo` between 0.186 and 0.209. Both Rosetta
arms sit below zero across the whole range, which the F4 notes attribute to the
mover filtering candidate mutations on the same score function that is being
plotted, together with its low hit rate; its ΔREU correlates with mutation count
at 0.07 against 0.37 to 0.83 for every other arm.

**Hydrogen bonding and h-bond protection.** On the 7 scaffolds carried by all
four arms of the protection comparison, protecting h-bonded sidechains retains
more strong side-chain hydrogen bonds than releasing protection, on both design
methods (F6, panel c). Against `mpnn_soluble`, `mpnn_halo` shows a median 2.0
fewer strong hydrogen bonds per structure (Holm p < 0.0001, Cliff's delta
−0.27), while `mpnn_hyper` shows no detectable difference (median 0, Holm
p = 1.0).

**Predicted structure.** Median ESMFold pLDDT is 77.4 for `mpnn_soluble`, 78.1
for `mpnn_hyper`, 78.2 for `mpnn_halo` and 78.6 for `mpnn_vanilla_weights`;
median ΔpLDDT against the matched wild-type prediction is −1.58, −1.33, −1.42
and −1.57 respectively. For comparison the random control sits at 74.4 and
−3.23, and `mpnn_soluble_hbond_protected` at 72.8 and −3.27, though the latter
runs on only 7 scaffolds. Neither biased arm differs detectably from
`mpnn_soluble` on ΔpLDDT after correction (Holm p = 0.07 and 0.30). Predicted
confidence falls with mutation count on every arm (F9, panel a), but 10 of the
25 wild types already predict below pLDDT 80 before any design is made (panel
b), so the absolute threshold reports scaffold choice more than design quality.

**Capacity of the biased checkpoints.** The two biased-weight arms reach every
target but need more sampling temperature to do it. The fraction of cells
reaching the target at the base temperature 0.3 is 0.89 for `mpnn_soluble` and
0.80 for `mpnn_vanilla_weights`, against 0.59 for `mpnn_hyper` and 0.58 for
`mpnn_halo`. At the most extreme rung, |ΔQ density| 24, the median final temperature is
0.30 for `mpnn_soluble`, 0.40 for `mpnn_vanilla_weights` and 0.60 for both
`mpnn_hyper` and `mpnn_halo`. F10 panel a plots the scaffold-balanced form of
this, the median of per-scaffold medians, which runs slightly lower for
`mpnn_halo` than the pooled figure quoted here.
This is a descriptive comparison: `final_temperature` is not one of the metrics
carried through the paired testing in `results/statistics.csv`, so no
corrected p-value accompanies it. Temperature and mutation count rise together
across the range, so panels b and c of F10 do not separate the two.

**Runtime.** Median CPU-seconds per on-target design is 0.94 for
`mpnn_soluble`, 0.97 for `mpnn_vanilla_weights`, 1.13 for `mpnn_hyper`, 1.78 for
`mpnn_halo`, 7.79 for `avnapsa`, 108.3 for `rosetta` and 92.5 for
`rosetta_hbond_off` (F8). All arms were timed on the same 4-CPU allocation on
the same partition so the comparison is like for like. The
unguided sampler costs a median 8.6 s per on-target design, but only over the 81
of 200 cells where it hit at all; the other 119 are censored lower bounds and
are excluded, which makes that figure a lower bound on the true cost.
